# Reinforcement Learning Trading Agent

## Objective

This notebook implements a Reinforcement Learning based Forex Trading Agent.

The agent learns an optimal trading strategy using historical Forex market data.

The workflow includes:

- Environment Creation
- State Representation
- BUY / SELL / HOLD Actions
- Reward Function
- Q-Learning
- Learned Trading Policy

The learned policy supports AI-assisted trading decisions.

## Reinforcement Learning Workflow

```
Historical Forex Data
          ↓
Feature Engineering
          ↓
Trading Environment
          ↓
State Representation
          ↓
Q-Learning Agent
          ↓
Trading Policy
          ↓
BUY / SELL / HOLD Decisions
```

In [1]:
import pandas as pd
import numpy as np

import random

In [2]:
price = pd.read_csv("../data/forex_price_data.csv")

price["DateTime"] = pd.to_datetime(price["DateTime"])

price.head()

,DateTime,PairID,PairName,Open,High,Low,Close,Volume,Spread
0,2023-01-01 00:00:00,1,EUR/USD,1.08103,1.08103,1.08019,1.08054,2651,0.9
1,2023-01-01 01:00:00,1,EUR/USD,1.08010,1.08043,1.07974,1.08028,2292,1.0
2,2023-01-01 02:00:00,1,EUR/USD,1.08036,1.08158,1.07961,1.08054,1796,1.2
3,2023-01-01 03:00:00,1,EUR/USD,1.08068,1.08126,1.07973,1.08088,4019,1.4
4,2023-01-01 04:00:00,1,EUR/USD,1.08042,1.08042,1.07967,1.08030,2207,1.6


In [3]:
price["Return"] = price["Close"].pct_change()

price = price.dropna().copy()

In [4]:
price["State"] = np.where(
    price["Return"] > 0,
    1,
    0
)

price[["Close","Return","State"]].head()

,Close,Return,State
1,1.08028,-0.000241,0
2,1.08054,0.000241,1
3,1.08088,0.000315,1
4,1.08030,-0.000537,0
5,1.07998,-0.000296,0


In [5]:
actions = {
    0: "BUY",
    1: "SELL",
    2: "HOLD"
}

print(actions)

{0: 'BUY', 1: 'SELL', 2: 'HOLD'}


In [6]:
actions = {
    0: "BUY",
    1: "SELL",
    2: "HOLD"
}

print(actions)

{0: 'BUY', 1: 'SELL', 2: 'HOLD'}


In [7]:
# Reward Function

def get_reward(entry_price, exit_price, action):

    # BUY
    if action == 0:
        return exit_price - entry_price

    # SELL
    elif action == 1:
        return entry_price - exit_price

    # HOLD
    else:
        return 0

In [11]:
from src.rl_environment import ForexEnvironment
from src.q_learning import QLearningAgent

In [12]:
# Create Environment

env = ForexEnvironment(price)

print("Environment Created Successfully")

Environment Created Successfully


In [14]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd

from src.feature_engineering import create_features
from src.rl_environment import ForexEnvironment
from src.q_learning import QLearningAgent

In [15]:
price = pd.read_csv("../data/forex_price_data.csv")

price = create_features(price)

In [16]:
env = ForexEnvironment(price)

print("Environment Created Successfully")

Environment Created Successfully


In [17]:
actions = ["BUY", "SELL", "HOLD"]

agent = QLearningAgent(actions)

print("Agent Created Successfully")

Agent Created Successfully


In [19]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd

from src.feature_engineering import create_features
from src.rl_environment import ForexEnvironment
from src.q_learning import QLearningAgent

In [20]:
price = pd.read_csv("../data/forex_price_data.csv")

price = create_features(price)

In [21]:
print(price.columns.tolist())

['DateTime', 'PairID', 'PairName', 'Open', 'High', 'Low', 'Close', 'Volume', 'Spread', 'MA_5', 'MA_20', 'Price_Change', 'High_Low_Range', 'Return', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Histogram', 'BB_Upper', 'BB_Middle', 'BB_Lower', 'Target', 'RSI_State', 'MACD_State', 'Trend_State']


In [22]:
env = ForexEnvironment(price)

actions = ["BUY", "SELL", "HOLD"]

agent = QLearningAgent(actions)

state = env.reset()

print(state)

(np.int64(1), np.int64(0), np.int64(0))


In [23]:
episodes = 100

for episode in range(episodes):

    state = env.reset()

    done = False

    while not done:

        # Choose Action
        action = agent.choose_action(state)

        # Execute Action
        next_state, reward, done = env.step(action)

        # Learn from Experience
        agent.learn(
            state,
            action,
            reward,
            next_state
        )

        # Move to Next State
        state = next_state

print("Training Completed!")

Training Completed!


In [24]:
print("Total Learned States:", len(agent.q_table))

# Show first 5 learned states
count = 0

for state, values in agent.q_table.items():
    print(f"State: {state} -> {values}")
    count += 1

    if count == 5:
        break

Total Learned States: 6
State: (np.int64(1), np.int64(0), np.int64(0)) -> [np.float64(0.0006456954174798129), np.float64(0.0004963081682510815), np.float64(0.0008850959691699105)]
State: (np.int64(1), np.int64(1), np.int64(0)) -> [np.float64(0.001592945937099401), np.float64(0.0009931870009100324), np.float64(0.0012555289233764336)]
State: (np.int64(1), np.int64(1), np.int64(1)) -> [np.float64(0.000883925949744661), np.float64(0.0007020324325668816), np.float64(0.00120271707385066)]
State: (np.int64(1), np.int64(0), np.int64(1)) -> [np.float64(0.001257806490417126), np.float64(0.00019254056911188348), np.float64(0.0007178530533505247)]
State: (np.int64(0), np.int64(0), np.int64(0)) -> [np.float64(0.0005706844326491714), np.float64(0.0006318845124639848), np.float64(0.0007518323947620947)]


In [25]:
action_names = ["BUY", "SELL", "HOLD"]

policy = {}

for state, q_values in agent.q_table.items():

    best_action = q_values.index(max(q_values))

    policy[state] = action_names[best_action]

print("Learned Trading Policy:\n")

count = 0

for state, action in policy.items():

    print(state, "->", action)

    count += 1

    if count == 10:
        break

Learned Trading Policy:

(np.int64(1), np.int64(0), np.int64(0)) -> HOLD
(np.int64(1), np.int64(1), np.int64(0)) -> BUY
(np.int64(1), np.int64(1), np.int64(1)) -> HOLD
(np.int64(1), np.int64(0), np.int64(1)) -> BUY
(np.int64(0), np.int64(0), np.int64(0)) -> HOLD
(np.int64(2), np.int64(1), np.int64(1)) -> SELL
